# Recompute Table 2 on all 50 samples

This notebook re-runs evaluation for all sample folders under `dataset/` and rebuilds **Table 2** for:
- Center-fan
- Planar + removed-part-aware
- Planar + largest-hole-only

Outputs:
- per-sample result CSVs for all 50 samples
- summary CSVs
- `table2_all50_summary.csv`
- `table2_all50_summary.md`

If your actual module/function names differ, edit only the **IMPORT YOUR PROJECT FUNCTIONS** cell.

In [1]:
import os
os.environ['KMP_DUPLICATE_LIB_OK'] = 'True'

from pathlib import Path
import sys
import traceback
import pandas as pd
import numpy as np
import open3d as o3d
import json

Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.


In [2]:
# ============================
# CONFIG
# ============================
PROJECT_ROOT = Path.cwd().parent
DATASET_DIR = PROJECT_ROOT / 'dataset'
CSV_DIR = PROJECT_ROOT / 'results' / 'csv'
CSV_DIR.mkdir(parents=True, exist_ok=True)

# Leave as None to evaluate all sample folders under dataset/
SAMPLE_IDS = None

OUT_CENTER_RESULTS = CSV_DIR / 'baseline_center_fan_results_all50.csv'
OUT_CENTER_SUMMARY = CSV_DIR / 'baseline_center_fan_summary_all50.csv'

OUT_PLANAR_RESULTS = CSV_DIR / 'baseline_planar_results_all50.csv'
OUT_PLANAR_SUMMARY = CSV_DIR / 'baseline_planar_summary_all50.csv'

OUT_LH_RESULTS = CSV_DIR / 'baseline_planar_largest_hole_results_all50.csv'
OUT_LH_SUMMARY = CSV_DIR / 'baseline_planar_largest_hole_summary_all50.csv'

OUT_TABLE2_CSV = CSV_DIR / 'table2_all50_summary.csv'
OUT_TABLE2_MD = CSV_DIR / 'table2_all50_summary.md'
PARTNET_ROOT = Path("E:/datasets/partnet/data_v0/data_v0")
print('PROJECT_ROOT =', PROJECT_ROOT)
print('DATASET_DIR   =', DATASET_DIR)
print('CSV_DIR       =', CSV_DIR)

PROJECT_ROOT = D:\MyJupyter\Works\3Dsegment
DATASET_DIR   = D:\MyJupyter\Works\3Dsegment\dataset
CSV_DIR       = D:\MyJupyter\Works\3Dsegment\results\csv


## IMPORT YOUR PROJECT FUNCTIONS

If these imports fail, only edit this cell to match your real code layout.

In [3]:
sys.path.append(str(PROJECT_ROOT))
sys.path.append(str(PROJECT_ROOT / 'baselines'))
sys.path.append(str(PROJECT_ROOT / 'evaluation'))

from baseline_center_fan import repair_one_sample_center_fan
from baseline_planar_triangulation import repair_one_sample_planar
from baseline_planar_largest_hole import repair_one_sample_planar_largest_hole

from baseline_planar_triangulation import (
    compute_total_loop_perimeter,
    count_new_vertices,
    compute_added_face_quality,
    compute_added_face_locality,
    get_boundary_edges_o3d,
    boundary_edges_to_loops,
    choose_nearby_loops_by_removed_part_from_mesh,
)

In [4]:
def read_mesh(mesh_path: Path):
    mesh = o3d.io.read_triangle_mesh(str(mesh_path))
    if mesh.is_empty():
        raise FileNotFoundError(f'Cannot read mesh: {mesh_path}')
    return mesh


def find_default_repaired_mesh(sample_dir: Path, method_tag: str):
    if method_tag == 'center_fan':
        candidates = [
            sample_dir / 'M_r_center_fan.obj',
            sample_dir / 'center_fan.obj',
            sample_dir / 'M_r.obj',
        ]
    elif method_tag == 'planar':
        candidates = [
            sample_dir / 'M_r_planar.obj',
            sample_dir / 'planar.obj',
            sample_dir / 'M_r.obj',
        ]
    elif method_tag == 'largest_hole':
        candidates = [
            sample_dir / 'M_r_planar_largest_hole.obj',
            sample_dir / 'M_r_planar_largest_hole_only.obj',
            sample_dir / 'planar_largest_hole_only.obj',
            sample_dir / 'largest_hole_only.obj',
            sample_dir / 'M_r.obj',
        ]
    else:
        candidates = []

    for p in candidates:
        if p.exists():
            return p
    return None


def run_repair_and_get_mesh_path(sample_dir: Path, method_tag: str):
    if method_tag == 'center_fan':
        ret = repair_one_sample_center_fan(sample_dir)
    elif method_tag == 'planar':
        ret = repair_one_sample_planar(sample_dir)
    elif method_tag == 'largest_hole':
        ret = repair_one_sample_planar_largest_hole(sample_dir)
    else:
        raise ValueError(method_tag)

    if isinstance(ret, (str, Path)):
        ret_path = Path(ret)
        if ret_path.exists():
            return ret_path

    guessed = find_default_repaired_mesh(sample_dir, method_tag)
    if guessed is None:
        raise FileNotFoundError(
            f'Could not find repaired mesh for {sample_dir.name} after running {method_tag}. '
            f'Please edit find_default_repaired_mesh().'
        )
    return guessed


def load_removed_part_mesh(sample_dir: Path, partnet_root: Path):
    meta_path = sample_dir / 'meta.json'
    meta = json.loads(meta_path.read_text(encoding='utf-8'))

    sample_id = str(meta.get('sample_id', sample_dir.name))
    removed_obj_files = meta['removed_obj_files']

    # 候选样本根目录
    candidate_roots = []
    source_dir = Path(meta.get('source_dir', ''))
    if str(source_dir) and source_dir.exists():
        candidate_roots.append(source_dir)

    sample_root = partnet_root / sample_id
    if sample_root.exists():
        candidate_roots.append(sample_root)

    # 去重
    uniq_roots = []
    seen = set()
    for r in candidate_roots:
        rr = str(r.resolve()) if r.exists() else str(r)
        if rr not in seen:
            uniq_roots.append(r)
            seen.add(rr)

    meshes = []
    found_paths = []

    for rel in removed_obj_files:
        rel_path = Path(rel)
        rel_name = rel_path.name
        hit = None

        for root in uniq_roots:
            # 1) 先尝试直接拼接
            p1 = root / rel
            if p1.exists():
                hit = p1
                break

            # 2) 再尝试 objs/ 子目录
            p2 = root / 'objs' / rel_name
            if p2.exists():
                hit = p2
                break

            # 3) 最后递归搜索同名文件
            matches = list(root.rglob(rel_name))
            if len(matches) > 0:
                hit = matches[0]
                break

        if hit is not None:
            found_paths.append(hit)
            m = o3d.io.read_triangle_mesh(str(hit))
            if not m.is_empty():
                meshes.append(m)

    print(f'[{sample_dir.name}] found removed obj paths:')
    for p in found_paths:
        print('  ', p)

    if len(meshes) == 0:
        raise FileNotFoundError(
            f'Failed to load removed part mesh for sample {sample_dir.name}. '
            f'Checked roots: {uniq_roots}, files: {removed_obj_files}'
        )

    merged = o3d.geometry.TriangleMesh()
    all_v = []
    all_f = []
    v_offset = 0

    for m in meshes:
        V = np.asarray(m.vertices)
        F = np.asarray(m.triangles)
        if len(V) == 0 or len(F) == 0:
            continue
        all_v.append(V)
        all_f.append(F + v_offset)
        v_offset += len(V)

    if len(all_v) == 0:
        raise FileNotFoundError(f'Removed part mesh is empty for sample {sample_dir.name}')

    merged.vertices = o3d.utility.Vector3dVector(np.vstack(all_v))
    merged.triangles = o3d.utility.Vector3iVector(np.vstack(all_f))
    merged.compute_vertex_normals()
    return merged


def compute_target_area_perimeter(mesh, sample_dir: Path):
    removed_mesh = load_removed_part_mesh(sample_dir, PARTNET_ROOT)

    boundary_edges = get_boundary_edges_o3d(mesh)
    loops = boundary_edges_to_loops(boundary_edges)

    target_loops = choose_nearby_loops_by_removed_part_from_mesh(
        loops,
        mesh,
        removed_mesh
    )

    if target_loops is None or len(target_loops) == 0:
        return 0.0, 0

    V = np.asarray(mesh.vertices)
    total_len = compute_total_loop_perimeter(V, target_loops)
    return float(total_len), len(target_loops)


def safe_mean(values):
    values = [v for v in values if v is not None and not pd.isna(v)]
    return float(np.mean(values)) if values else np.nan


def evaluate_one_sample(sample_dir: Path, method_tag: str):
    md_path = sample_dir / 'M_d.obj'
    if not md_path.exists():
        raise FileNotFoundError(f'Missing M_d.obj: {md_path}')

    md_mesh = read_mesh(md_path)
    total_before, n_target_loops_before = compute_target_area_perimeter(md_mesh, sample_dir)

    mr_path = run_repair_and_get_mesh_path(sample_dir, method_tag)
    mr_mesh = read_mesh(mr_path)

    total_after, n_target_loops_after = compute_target_area_perimeter(mr_mesh, sample_dir)
    improvement = float(total_before - total_after)

    num_new_vertices = count_new_vertices(md_mesh, mr_mesh)
    num_added_faces = int(len(np.asarray(mr_mesh.triangles)) - len(np.asarray(md_mesh.triangles)))
    mean_added_face_quality, min_added_face_quality = compute_added_face_quality(md_mesh, mr_mesh)

    locality = compute_added_face_locality(md_mesh, mr_mesh, sample_dir)

    return {
        'sample_id': sample_dir.name,
        'method': method_tag,
        'total_loop_len_before': total_before,
        'nearest_loop_len_after': total_after,
        'improvement': improvement,
        'num_target_loops_before': n_target_loops_before,
        'num_target_loops_after': n_target_loops_after,
        'num_new_vertices': num_new_vertices,
        'num_added_faces': num_added_faces,
        'mean_added_face_quality': mean_added_face_quality,
        'min_added_face_quality': min_added_face_quality,
        'num_added_faces_inside_zone': locality.get('num_added_faces_inside_zone', np.nan),
        'num_added_faces_outside_zone': locality.get('num_added_faces_outside_zone', np.nan),
        'face_locality_ratio': locality.get('face_locality_ratio', np.nan),
        'repaired_mesh_path': str(mr_path),
    }


def summarize_results(df: pd.DataFrame, method_name: str):
    required_cols = [
        'total_loop_len_before',
        'nearest_loop_len_after',
        'improvement',
        'num_new_vertices',
        'num_added_faces',
        'mean_added_face_quality',
        'min_added_face_quality',
        'num_added_faces_inside_zone',
        'num_added_faces_outside_zone',
        'face_locality_ratio',
    ]

    if df is None or len(df) == 0:
        return pd.DataFrame([{
            'method': method_name,
            'n_samples': 0,
            'mean_total_loop_len_before': np.nan,
            'mean_target_loop_len_after': np.nan,
            'mean_improvement': np.nan,
            'mean_num_new_vertices': np.nan,
            'mean_num_added_faces': np.nan,
            'mean_added_face_quality': np.nan,
            'mean_min_added_face_quality': np.nan,
            'mean_added_faces_inside_zone': np.nan,
            'mean_added_faces_outside_zone': np.nan,
            'mean_face_locality_ratio': np.nan,
        }])

    for c in required_cols:
        if c not in df.columns:
            df[c] = np.nan

    return pd.DataFrame([{
        'method': method_name,
        'n_samples': len(df),
        'mean_total_loop_len_before': safe_mean(df['total_loop_len_before']),
        'mean_target_loop_len_after': safe_mean(df['nearest_loop_len_after']),
        'mean_improvement': safe_mean(df['improvement']),
        'mean_num_new_vertices': safe_mean(df['num_new_vertices']),
        'mean_num_added_faces': safe_mean(df['num_added_faces']),
        'mean_added_face_quality': safe_mean(df['mean_added_face_quality']),
        'mean_min_added_face_quality': safe_mean(df['min_added_face_quality']),
        'mean_added_faces_inside_zone': safe_mean(df['num_added_faces_inside_zone']),
        'mean_added_faces_outside_zone': safe_mean(df['num_added_faces_outside_zone']),
        'mean_face_locality_ratio': safe_mean(df['face_locality_ratio']),
    }])
def evaluate_one_sample(sample_dir: Path, method_tag: str):
    if method_tag == 'center_fan':
        ret = repair_one_sample_center_fan(sample_dir)
    elif method_tag == 'planar':
        ret = repair_one_sample_planar(sample_dir)
    elif method_tag == 'largest_hole':
        ret = repair_one_sample_planar_largest_hole(sample_dir)
    else:
        raise ValueError(method_tag)

    if not isinstance(ret, dict):
        raise TypeError(f'Expected dict return from {method_tag}, got: {type(ret)}')

    if not ret.get('success', False):
        raise RuntimeError(f"{method_tag} failed on {sample_dir.name}: {ret}")

    row = {
        'sample_id': str(ret.get('sample_id', sample_dir.name)),
        'method': method_tag,
        'total_loop_len_before': ret.get('total_loop_len_before', np.nan),
        'nearest_loop_len_after': ret.get('nearest_loop_len_after', np.nan),
        'improvement': ret.get('improvement', np.nan),
        'num_target_loops_before': len(ret.get('nearby_loops', [])) if ret.get('nearby_loops', None) is not None else np.nan,
        'num_target_loops_after': np.nan,  # 如果返回值没有，就先留空
        'num_new_vertices': ret.get('num_new_vertices', np.nan),
        'num_added_faces': ret.get('num_added_faces', np.nan),
        'mean_added_face_quality': ret.get('mean_added_face_quality', np.nan),
        'min_added_face_quality': ret.get('min_added_face_quality', np.nan),
        'num_added_faces_inside_zone': ret.get('num_added_faces_inside_zone', np.nan),
        'num_added_faces_outside_zone': ret.get('num_added_faces_outside_zone', np.nan),
        'face_locality_ratio': ret.get('face_locality_ratio', np.nan),
        'repaired_mesh_path': np.nan,
    }
    return row

def list_sample_dirs():
    if SAMPLE_IDS is None:
        sample_dirs = sorted([p for p in DATASET_DIR.iterdir() if p.is_dir()])
    else:
        sample_dirs = [DATASET_DIR / sid for sid in SAMPLE_IDS]
    return [p for p in sample_dirs if p.exists()]


def evaluate_method_all_samples(method_tag: str):
    rows, failed = [], []
    sample_dirs = list_sample_dirs()
    print(f'\n[{method_tag}] evaluating {len(sample_dirs)} samples ...')

    for i, sample_dir in enumerate(sample_dirs, start=1):
        try:
            row = evaluate_one_sample(sample_dir, method_tag)
            rows.append(row)
            print(f'[{method_tag}] {i:03d}/{len(sample_dirs)} OK  {sample_dir.name}')
        except Exception as e:
            failed.append({'sample_id': sample_dir.name, 'error': repr(e)})
            print(f'[{method_tag}] {i:03d}/{len(sample_dirs)} FAIL {sample_dir.name}: {e}')
            traceback.print_exc()

    return pd.DataFrame(rows), pd.DataFrame(failed)

## Run all three methods on all samples

In [5]:
center_df, center_fail_df = evaluate_method_all_samples('center_fan')
planar_df, planar_fail_df = evaluate_method_all_samples('planar')
lh_df, lh_fail_df = evaluate_method_all_samples('largest_hole')


[center_fan] evaluating 100 samples ...
[center_fan] 001/100 OK  2101
[center_fan] 002/100 OK  2168
[center_fan] 003/100 OK  2210
[center_fan] 004/100 OK  2269
[center_fan] 005/100 OK  2398
[center_fan] 006/100 OK  2549
[center_fan] 007/100 OK  2594
[center_fan] 008/100 OK  2634
[center_fan] 009/100 OK  2689
[center_fan] 010/100 OK  3022
[center_fan] 011/100 OK  3225
[center_fan] 012/100 OK  3278
[center_fan] 013/100 OK  35109
[center_fan] 014/100 OK  35185
[center_fan] 015/100 OK  35233
[center_fan] 016/100 OK  35388
[center_fan] 017/100 OK  35396
[center_fan] 018/100 OK  35580
[center_fan] 019/100 OK  35618
[center_fan] 020/100 OK  35691
[center_fan] 021/100 OK  35871
[center_fan] 022/100 OK  35916
[center_fan] 023/100 OK  35923
[center_fan] 024/100 OK  36006
[center_fan] 025/100 OK  36074
[center_fan] 026/100 OK  36081
[center_fan] 027/100 OK  36192
[center_fan] 028/100 OK  36317
[center_fan] 029/100 OK  36402
[center_fan] 030/100 OK  36460
[center_fan] 031/100 OK  36730
[center_fa

In [29]:
# Save per-sample results
center_df.to_csv(OUT_CENTER_RESULTS, index=False)
planar_df.to_csv(OUT_PLANAR_RESULTS, index=False)
lh_df.to_csv(OUT_LH_RESULTS, index=False)

center_fail_df.to_csv(CSV_DIR / 'baseline_center_fan_failed_all50.csv', index=False)
planar_fail_df.to_csv(CSV_DIR / 'baseline_planar_failed_all50.csv', index=False)
lh_fail_df.to_csv(CSV_DIR / 'baseline_planar_largest_hole_failed_all50.csv', index=False)

print('Saved per-sample result CSVs.')
print(OUT_CENTER_RESULTS)
print(OUT_PLANAR_RESULTS)
print(OUT_LH_RESULTS)

Saved per-sample result CSVs.
D:\MyJupyter\Works\3Dsegment\results\csv\baseline_center_fan_results_all50.csv
D:\MyJupyter\Works\3Dsegment\results\csv\baseline_planar_results_all50.csv
D:\MyJupyter\Works\3Dsegment\results\csv\baseline_planar_largest_hole_results_all50.csv


In [30]:
# Summaries
center_summary = summarize_results(center_df, 'Center-fan')
planar_summary = summarize_results(planar_df, 'Planar + removed-part-aware')
lh_summary = summarize_results(lh_df, 'Planar + largest-hole-only')

center_summary.to_csv(OUT_CENTER_SUMMARY, index=False)
planar_summary.to_csv(OUT_PLANAR_SUMMARY, index=False)
lh_summary.to_csv(OUT_LH_SUMMARY, index=False)

print('Saved summary CSVs.')
print(OUT_CENTER_SUMMARY)
print(OUT_PLANAR_SUMMARY)
print(OUT_LH_SUMMARY)

Saved summary CSVs.
D:\MyJupyter\Works\3Dsegment\results\csv\baseline_center_fan_summary_all50.csv
D:\MyJupyter\Works\3Dsegment\results\csv\baseline_planar_summary_all50.csv
D:\MyJupyter\Works\3Dsegment\results\csv\baseline_planar_largest_hole_summary_all50.csv


In [31]:
# Build Table 2
table2 = pd.DataFrame([
    {
        'Method': 'Center-fan',
        'n_samples': int(center_summary.iloc[0]['n_samples']) if len(center_summary) else 0,
        'Mean target loop length after repair ↓': center_summary.iloc[0]['mean_target_loop_len_after'] if len(center_summary) else np.nan,
        'Mean improvement ↑': center_summary.iloc[0]['mean_improvement'] if len(center_summary) else np.nan,
        'Mean new vertices ↓': center_summary.iloc[0]['mean_num_new_vertices'] if len(center_summary) else np.nan,
        'Mean added faces ↓': center_summary.iloc[0]['mean_num_added_faces'] if len(center_summary) else np.nan,
        'Mean added-face quality ↑': center_summary.iloc[0]['mean_added_face_quality'] if len(center_summary) else np.nan,
        'Mean minimum added-face quality ↑': center_summary.iloc[0]['mean_min_added_face_quality'] if len(center_summary) else np.nan,
    },
    {
        'Method': 'Planar + removed-part-aware',
        'n_samples': int(planar_summary.iloc[0]['n_samples']) if len(planar_summary) else 0,
        'Mean target loop length after repair ↓': planar_summary.iloc[0]['mean_target_loop_len_after'] if len(planar_summary) else np.nan,
        'Mean improvement ↑': planar_summary.iloc[0]['mean_improvement'] if len(planar_summary) else np.nan,
        'Mean new vertices ↓': planar_summary.iloc[0]['mean_num_new_vertices'] if len(planar_summary) else np.nan,
        'Mean added faces ↓': planar_summary.iloc[0]['mean_num_added_faces'] if len(planar_summary) else np.nan,
        'Mean added-face quality ↑': planar_summary.iloc[0]['mean_added_face_quality'] if len(planar_summary) else np.nan,
        'Mean minimum added-face quality ↑': planar_summary.iloc[0]['mean_min_added_face_quality'] if len(planar_summary) else np.nan,
    },
    {
        'Method': 'Planar + largest-hole-only',
        'n_samples': int(lh_summary.iloc[0]['n_samples']) if len(lh_summary) else 0,
        'Mean target loop length after repair ↓': lh_summary.iloc[0]['mean_target_loop_len_after'] if len(lh_summary) else np.nan,
        'Mean improvement ↑': lh_summary.iloc[0]['mean_improvement'] if len(lh_summary) else np.nan,
        'Mean new vertices ↓': lh_summary.iloc[0]['mean_num_new_vertices'] if len(lh_summary) else np.nan,
        'Mean added faces ↓': lh_summary.iloc[0]['mean_num_added_faces'] if len(lh_summary) else np.nan,
        'Mean added-face quality ↑': lh_summary.iloc[0]['mean_added_face_quality'] if len(lh_summary) else np.nan,
        'Mean minimum added-face quality ↑': lh_summary.iloc[0]['mean_min_added_face_quality'] if len(lh_summary) else np.nan,
    },
])

table2.to_csv(OUT_TABLE2_CSV, index=False)

display_df = table2.copy()
for c in display_df.columns:
    if c != 'Method':
        display_df[c] = display_df[c].map(
            lambda x: '' if pd.isna(x) else (f'{x:.6f}' if isinstance(x, float) and abs(x) < 10 else f'{x:.3f}' if isinstance(x, float) else str(x))
        )

md_text = display_df.to_markdown(index=False)
OUT_TABLE2_MD.write_text(md_text, encoding='utf-8')

print('Saved Table 2:')
print(OUT_TABLE2_CSV)
print(OUT_TABLE2_MD)
print('\nTable 2 preview:')
print(md_text)

Saved Table 2:
D:\MyJupyter\Works\3Dsegment\results\csv\table2_all50_summary.csv
D:\MyJupyter\Works\3Dsegment\results\csv\table2_all50_summary.md

Table 2 preview:
| Method                      |   n_samples |   Mean target loop length after repair ↓ |   Mean improvement ↑ |   Mean new vertices ↓ |   Mean added faces ↓ |   Mean added-face quality ↑ |   Mean minimum added-face quality ↑ |
|:----------------------------|------------:|-----------------------------------------:|---------------------:|----------------------:|---------------------:|----------------------------:|------------------------------------:|
| Center-fan                  |         100 |                                 0        |             1.15916  |                  1.78 |               88.989 |                    0.377754 |                            0.181679 |
| Planar + removed-part-aware |         100 |                                 0.169078 |             0.990081 |                  0    |               77.37

In [26]:
sample_dir = DATASET_DIR / '2168'
ret = repair_one_sample_center_fan(sample_dir)
print('return value =', ret)

for p in sample_dir.glob('*'):
    print(p.name)

return value = {'sample_id': '2168', 'success': True, 'M_d': TriangleMesh with 74913 points and 150562 triangles., 'M_r': TriangleMesh with 74914 points and 150691 triangles., 'nearby_loops': [[49126, 49129, 49134, 49135, 49136, 49147, 49139, 49137, 49140, 49165, 49166, 49164, 49162, 49163, 49216, 49221, 49217, 49218, 49222, 49225, 49226, 49246, 49247, 49249, 49253, 49463, 49465, 50202, 50204, 50209, 50211, 50220, 50222, 50243, 50245, 50286, 50288, 50293, 50296, 50305, 50306, 50307, 50315, 50326, 50329, 50332, 50336, 50337, 50343, 50352, 50353, 50354, 50355, 50364, 50365, 50366, 50367, 52032, 52033, 52034, 52035, 52046, 52047, 52039, 52041, 52044, 52056, 52060, 52069, 52067, 52068, 52070, 52064, 52063, 51985, 51983, 52015, 52013, 52014, 52011, 52002, 51999, 51994, 51992, 51942, 51940, 51919, 51917, 51908, 51906, 51901, 51899, 51399, 51397, 51155, 51158, 51157, 51153, 51151, 51147, 51145, 51134, 51120, 51117, 51115, 51108, 51090, 51087, 51085, 50995, 50978, 50979, 50977, 50976, 50974, 5